In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, when

flight_schema = StructType([
    StructField("flight_id", StringType(), True),
    StructField("airline", StringType(), True),
    StructField("from_city", StringType(), True),
    StructField("to_city", StringType(), True),
    StructField("duration", IntegerType(), True),
    StructField("status", StringType(), True)
])

booking_schema = StructType([
    StructField("booking_id", StringType(), True),
    StructField("flight_id", StringType(), True),
    StructField("passenger_name", StringType(), True),
    StructField("travel_class", StringType(), True),
    StructField("ticket_price", IntegerType(), True),
    StructField("booking_date", StringType(), True)
])

flights_data = [
    ("F101", "Indigo", "Hyderabad", "Delhi", 140, "On Time"),
    ("F102", "Air India", "Mumbai", "Chennai", 120, "Delayed"),
    ("F103", "Vistara", "Bangalore", "Hyderabad", 90, "On Time"),
    ("F104", "Indigo", "Delhi", "Mumbai", 130, "Cancelled"),
    ("F105", "Air India", "Chennai", "Bangalore", 80, "On Time"),
    ("F106", "Akasa", "Pune", "Delhi", 150, "Delayed"),
    ("F107", "Vistara", "Hyderabad", "Kolkata", 160, "On Time"),
    ("F108", "Indigo", "Mumbai", "Hyderabad", 110, "On Time"),
    ("F109", "Akasa", "Delhi", "Chennai", 145, "Delayed"),
    ("F110", "Air India", "Bangalore", "Mumbai", 95, "On Time"),
    ("F111", "Indigo", "Hyderabad", "Goa", 75, "On Time"),
    ("F112", "Vistara", "Goa", "Delhi", 150, "Cancelled"),
    ("F113", "Akasa", "Chennai", "Pune", 100, "On Time"),
    ("F114", "Air India", "Kolkata", "Bangalore", 170, "Delayed"),
    ("F115", "Indigo", "Delhi", "Hyderabad", 135, "On Time"),
]

bookings_data = [
    ("B1001", "F101", "Rahul Sharma", "Economy", 8500, "2026-06-01"),
    ("B1002", "F101", "Priya Reddy", "Business", 22000, "2026-06-01"),
    ("B1003", "F102", "Amit Kumar", "Economy", 9000, "2026-06-02"),
    ("B1004", "F103", "Sneha Patel", "Premium Economy", 15000, "2026-06-02"),
    ("B1005", "F104", "Farhan Ali", "Economy", 7500, "2026-06-03"),
    ("B1006", "F105", "Neha Singh", "Business", 25000, "2026-06-03"),
    ("B1007", "F106", "Arjun Verma", "Economy", 10000, "2026-06-04"),
    ("B1008", "F107", "Meera Nair", "Premium Economy", 17000, "2026-06-04"),
    ("B1009", "F108", "Kiran Rao", "Economy", 9500, "2026-06-05"),
    ("B1010", "F109", "Nisha Reddy", "Business", 28000, "2026-06-05"),
    ("B1011", "F110", "David Thomas", "Economy", 8000, "2026-06-06"),
    ("B1012", "F111", "Ayesha Khan", "Premium Economy", 16000, "2026-06-06"),
    ("B1013", "F112", "Rohit Sharma", "Economy", 7000, "2026-06-07"),
    ("B1014", "F113", "Pooja Mehta", "Business", 24000, "2026-06-07"),
    ("B1015", "F114", "Sanjay Gupta", "Economy", 10500, "2026-06-08"),
    ("B1016", "F115", "Divya Iyer", "Premium Economy", 18000, "2026-06-08"),
    ("B1017", "F101", "Rahul Sharma", "Economy", 8500, "2026-06-09"),
    ("B1018", "F103", "Priya Reddy", "Business", 23000, "2026-06-09"),
    ("B1019", "F107", "Amit Kumar", "Economy", 9500, "2026-06-10"),
    ("B1020", "F110", "Sneha Patel", "Premium Economy", 15500, "2026-06-10"),
]

df_flights = spark.createDataFrame(flights_data, flight_schema)
df_bookings = spark.createDataFrame(bookings_data, booking_schema)
df_joined = df_bookings.join(df_flights, on="flight_id", how="left")
df_transformed = df_joined.withColumn("revenue", col("ticket_price")) \
    .withColumn("price_band",
        when(col("ticket_price") > 20000, "Premium")
        .when(col("ticket_price") > 10000, "Standard")
        .otherwise("Budget")
    ) \
    .withColumn("delay_flag",
        when(col("status") == "Delayed", "Yes")
        .otherwise("No")
    )
df_transformed.createOrReplaceTempView("transformed_data")

In [0]:
spark.sql("SELECT airline, SUM(revenue) AS total_revenue FROM transformed_data GROUP BY airline ORDER BY total_revenue DESC").show()

+---------+-------------+
|  airline|total_revenue|
+---------+-------------+
|   Indigo|        90000|
|  Vistara|        71500|
|Air India|        68000|
|    Akasa|        62000|
+---------+-------------+



In [0]:
spark.sql("SELECT travel_class, SUM(revenue) AS total_revenue FROM transformed_data GROUP BY travel_class ORDER BY total_revenue DESC").show()

+---------------+-------------+
|   travel_class|total_revenue|
+---------------+-------------+
|       Business|       122000|
|        Economy|        88000|
|Premium Economy|        81500|
+---------------+-------------+



In [0]:
spark.sql("SELECT status, COUNT(*) AS total FROM transformed_data GROUP BY status").show()

+---------+-----+
|   status|total|
+---------+-----+
|  On Time|   14|
|  Delayed|    4|
|Cancelled|    2|
+---------+-----+



In [0]:
spark.sql("SELECT to_city, COUNT(*) AS total_passengers FROM transformed_data GROUP BY to_city ORDER BY total_passengers DESC LIMIT 5").show()

+---------+----------------+
|  to_city|total_passengers|
+---------+----------------+
|    Delhi|               5|
|Hyderabad|               4|
|   Mumbai|               3|
|  Chennai|               2|
|Bangalore|               2|
+---------+----------------+

